<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/05_Hands_On_Labs/notebooks/07_model_evaluation_and_explainability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 07: Model Evaluation & Explainability

> **Goal:** Go beyond accuracy — learn the full toolkit for evaluating ML models and explaining their predictions.

**What you'll learn:**
1. Classification metrics in depth (precision, recall, F1, AUC-ROC, AUC-PR)
2. When to use which metric
3. Calibration — are your probabilities trustworthy?
4. SHAP for model explainability
5. Threshold optimization for business objectives

**Time:** ~3 hours

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
    roc_curve, auc, roc_auc_score,
    precision_recall_curve, average_precision_score,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print('Imports successful!')

## Part 1: The Confusion Matrix — Everything Flows From This

In [ ]:
# Simulated fraud detection scenario
# Class imbalance: 2% fraud rate
X, y = make_classification(
    n_samples=10000,
    n_features=20,
    n_informative=15,
    weights=[0.98, 0.02],  # 98% non-fraud, 2% fraud
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train fraud rate: {y_train.mean():.3f} ({y_train.sum()} frauds)')
print(f'Test fraud rate:  {y_test.mean():.3f} ({y_test.sum()} frauds)')

# Train model
model = GradientBoostingClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

In [ ]:
# Confusion matrix anatomy
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('CONFUSION MATRIX BREAKDOWN')
print('=' * 50)
print(f'True Negatives  (TN): {tn:5d}  → Legit txn, predicted legit ✅')
print(f'False Positives (FP): {fp:5d}  → Legit txn, predicted fraud ❌ (Type I error)')
print(f'False Negatives (FN): {fn:5d}  → Fraud, predicted legit ❌ (Type II error)')
print(f'True Positives  (TP): {tp:5d}  → Fraud, predicted fraud ✅')
print()

# Derived metrics
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
accuracy = (tp + tn) / (tp + tn + fp + fn)
specificity = tn / (tn + fp)

print('DERIVED METRICS')
print(f'Accuracy:    {accuracy:.4f}  (misleading for imbalanced data!)')
print(f'Precision:   {precision:.4f}  (of predicted frauds, how many real?)')
print(f'Recall:      {recall:.4f}  (of real frauds, how many caught?)')
print(f'F1 Score:    {f1:.4f}  (harmonic mean of precision & recall)')
print(f'Specificity: {specificity:.4f}  (true negative rate)')

# Visualize
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Legit', 'Fraud'],
    cmap='Blues',
    ax=ax
)
ax.set_title('Confusion Matrix — Fraud Detection')
plt.tight_layout()
plt.show()

## Part 2: ROC-AUC vs PR-AUC — When to Use Which?

In [ ]:
# ROC-AUC: Good for balanced datasets
# PR-AUC: Better for imbalanced datasets (fraud, medical, spam)
# Rule: If positive class is rare (<10%), PR-AUC is more informative

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── ROC Curve ─────────────────────────────────────────────────
ax = axes[0]
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

ax.plot(fpr, tpr, 'b-', lw=2, label=f'GBT (AUC={roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.500)')
ax.fill_between(fpr, tpr, alpha=0.1, color='blue')
ax.set_xlabel('False Positive Rate (FPR)')
ax.set_ylabel('True Positive Rate (TPR = Recall)')
ax.set_title('ROC Curve')
ax.legend()

# ── Precision-Recall Curve ─────────────────────────────────────
ax = axes[1]
precision_vals, recall_vals, pr_thresholds = precision_recall_curve(y_test, y_proba)
avg_precision = average_precision_score(y_test, y_proba)
baseline = y_test.mean()  # Random classifier PR-AUC ≈ fraud rate

ax.plot(recall_vals, precision_vals, 'g-', lw=2, label=f'GBT (AP={avg_precision:.3f})')
ax.axhline(baseline, color='red', linestyle='--', lw=1,
           label=f'Random classifier ({baseline:.3f})')
ax.fill_between(recall_vals, precision_vals, alpha=0.1, color='green')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend()

plt.suptitle('ROC vs Precision-Recall (Imbalanced Dataset: 2% fraud)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'ROC-AUC:  {roc_auc:.4f}  (looks great! But maybe misleading)')
print(f'PR-AUC:   {avg_precision:.4f}  (more realistic for imbalanced data)')
print(f'Baseline: {baseline:.4f}  (random classifier PR-AUC for 2% fraud rate)')

## Part 3: Threshold Optimization for Business Objectives

In [ ]:
# Business context:
# - Average fraud loss if missed: $500
# - Cost to investigate a flagged transaction: $10
# - We want to maximize expected profit

FRAUD_COST = 500      # Loss from each missed fraud (false negative)
REVIEW_COST = 10      # Cost to review each flagged transaction (includes FP)

thresholds_to_test = np.linspace(0.01, 0.99, 100)
results = []

for threshold in thresholds_to_test:
    y_pred_thresh = (y_proba >= threshold).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_thresh)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()

    prec = tp_t / (tp_t + fp_t + 1e-8)
    rec = tp_t / (tp_t + fn_t + 1e-8)

    # Expected cost (lower is better)
    cost = (fn_t * FRAUD_COST) + ((tp_t + fp_t) * REVIEW_COST)

    results.append({
        'threshold': threshold,
        'precision': prec,
        'recall': rec,
        'f1': 2 * prec * rec / (prec + rec + 1e-8),
        'total_cost': cost,
        'flagged': tp_t + fp_t,
    })

df_thresh = pd.DataFrame(results)

# Find optimal thresholds
best_f1 = df_thresh.loc[df_thresh['f1'].idxmax()]
best_cost = df_thresh.loc[df_thresh['total_cost'].idxmin()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision-Recall vs threshold
axes[0].plot(df_thresh['threshold'], df_thresh['precision'], 'b-', label='Precision')
axes[0].plot(df_thresh['threshold'], df_thresh['recall'], 'g-', label='Recall')
axes[0].plot(df_thresh['threshold'], df_thresh['f1'], 'r-', lw=2, label='F1 Score')
axes[0].axvline(best_f1['threshold'], color='red', linestyle='--', alpha=0.7,
                label=f'Best F1 threshold: {best_f1["threshold"]:.2f}')
axes[0].set_xlabel('Decision Threshold')
axes[0].set_ylabel('Score')
axes[0].set_title('Metrics vs Threshold')
axes[0].legend()

# Cost vs threshold
axes[1].plot(df_thresh['threshold'], df_thresh['total_cost'], 'purple', lw=2)
axes[1].axvline(best_cost['threshold'], color='green', linestyle='--', alpha=0.7,
                label=f'Min cost threshold: {best_cost["threshold"]:.2f}')
axes[1].set_xlabel('Decision Threshold')
axes[1].set_ylabel('Total Cost ($)')
axes[1].set_title('Business Cost vs Threshold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Best F1 threshold: {best_f1["threshold"]:.2f}')
print(f'  Precision: {best_f1["precision"]:.3f}, Recall: {best_f1["recall"]:.3f}')
print()
print(f'Best cost threshold: {best_cost["threshold"]:.2f}')
print(f'  Total cost: ${best_cost["total_cost"]:,.0f}')
print(f'  Flagged transactions: {best_cost["flagged"]:.0f}')

## Part 4: SHAP — Model Explainability

In [ ]:
# pip install shap
try:
    import shap

    # SHAP for GBT model
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test[:500])  # Sample for speed

    # Feature names
    feature_names = [f'feature_{i}' for i in range(X.shape[1])]

    # Summary plot — global feature importance
    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        shap_values, X_test[:500],
        feature_names=feature_names,
        show=False,
        max_display=10,
    )
    plt.title('SHAP Feature Importance (Global)')
    plt.tight_layout()
    plt.show()

    print('SHAP interpretation:')
    print('- Each dot = one prediction')
    print('- Red dots = high feature value')
    print('- Blue dots = low feature value')
    print('- Right = increases fraud probability')
    print('- Left = decreases fraud probability')

except ImportError:
    print('shap not installed. Run: pip install shap')
    print()
    print('Alternative: sklearn feature importances (less accurate)')
    importance = pd.Series(
        model.feature_importances_,
        index=[f'feature_{i}' for i in range(X.shape[1])]
    ).sort_values(ascending=False)

    plt.figure(figsize=(8, 5))
    importance[:10].plot(kind='barh', color='steelblue')
    plt.xlabel('Feature Importance (Gini)')
    plt.title('Top 10 Feature Importances')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## Summary: Which Metric to Use?

| Scenario | Metric | Reason |
|----------|--------|--------|
| Balanced classes | Accuracy, F1 | Both classes equally important |
| Imbalanced, FP costly | Precision | Spam detection, medical false alarms |
| Imbalanced, FN costly | Recall | Fraud, disease screening |
| Ranking/probability quality | AUC-ROC | Threshold-agnostic overall performance |
| Imbalanced + probability | AUC-PR | Better than ROC for rare positive classes |
| Business optimization | Custom cost | Threshold set to minimize expected loss |

## Challenge
1. Train a logistic regression and random forest on the same data. Compare their calibration curves.
2. Apply SMOTE oversampling and see if PR-AUC improves.
3. Write a function that takes business costs (FP cost, FN cost) and returns the optimal threshold.